In [15]:
from PIL import Image, ImageDraw, ImageFont

# =========================
# EINSTELLUNGEN
# =========================
title_text = "GOAT results"
bottom_text = "PBE0 def2-TZVP D3"

# Bilder der ersten Reihe (5 Stück)
row1 = [
    ("benz_opt.dat.tga", "opt", "0.0 kJ/mol"),
    ("benz_pd.dat.tga", "pd", "-2.1 kJ/mol"),
    ("benz_s_new.dat.tga", "s", "-2.4 kJ/mol"),
    ("benz_t.dat.tga", "t", "-2.6 kJ/mol"),
    ("benz_x.dat.tga", "x", "5.8 kJ/mol"),
]

# Bilder der zweiten Reihe (6 Stück)
row2 = [
    ("f6_opt.dat.tga", "opt", "0.0 kJ/mol"),
    ("f6_sandwich.dat.tga", "pd", "0.1 kJ/mol"),
    ("f6_t_new.dat.tga", "s", "10.8 kJ/mol"),
    ("f6_t_new_geom.dat.tga", "t", "11.8 kJ/mol"),
    ("f6_x.dat.tga", "x", "17.3 kJ/mol"),
]

# Ausgabeparameter
width_cm = 16
dpi = 300
output_file = "übersicht.png"

# =========================
# BERECHNUNGEN
# =========================

width_px = int(width_cm / 2.54 * dpi)

cols = max(len(row1), len(row2))
padding = 40
label_space = 70

cell_width = (width_px - padding*(cols+1)) // cols

# Bilder laden und skalieren
def load_and_resize(path):
    img = Image.open(path)
    ratio = cell_width / img.width
    new_height = int(img.height * ratio)
    return img.resize((cell_width, new_height))

imgs_row1 = [load_and_resize(i[0]) for i in row1]
imgs_row2 = [load_and_resize(i[0]) for i in row2]

max_h1 = max(i.height for i in imgs_row1)
max_h2 = max(i.height for i in imgs_row2)

title_space = 30
bottom_space = 70
line_spacing = 60      # Abstand Label → Energy
text_offset = 5        # Abstand Bild → Label
text_block = text_offset + line_spacing + 40

height_px = (
    title_space +
    padding +
    max_h1 + text_block +
    padding +
    max_h2 + text_block +
    bottom_space +
    padding
)

canvas = Image.new("RGB", (width_px, height_px), "white")
draw = ImageDraw.Draw(canvas)

# Schrift
font = ImageFont.load_default()
font_title = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf", 60)
font_label = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf", 36)
font_energy = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf", 30)
# ZEICHNEN
# =========================

def draw_row(row, imgs, y_start):
    for i, ((path, label, energy), img) in enumerate(zip(row, imgs)):
        x = padding + i*(cell_width + padding)

        canvas.paste(img, (x, y_start))

        text_y = y_start + img.height + 5

        w = draw.textlength(title_text, font=font_title)
        draw.text(((width_px - w)/2, padding), title_text, fill="black", font=font_title)

        w = draw.textlength(label, font=font_label)
        draw.text((x + (cell_width-w)/2, text_y), label, fill="black", font=font_label)

        w = draw.textlength(energy, font=font_energy)
        draw.text((x + (cell_width-w)/2, text_y + 50), energy, fill="black", font=font_energy)

# erste Reihe
row1_y = title_space + padding
draw_row(row1, imgs_row1, row1_y)

# zweite Reihe
row2_y = row1_y + max_h1 + text_block + padding
draw_row(row2, imgs_row2, row2_y)

w = draw.textlength(bottom_text, font=font_energy)

draw.text(
    ((width_px - w)/2, height_px - bottom_space),
    bottom_text,
    fill="black",
    font=font_energy
)

# speichern
canvas.save(output_file, dpi=(dpi, dpi))

print("Bild gespeichert:", output_file)

Bild gespeichert: übersicht.png
